# Week 7: Fine-Tuning an LLM for Transaction Risk Scoring using an opensource LLM

---

## Details

An exercise to fine-tune `meta-llama/Llama-3.2-1B-Instruct` on a real-world AML (Anti-Money Laundering) dataset
using **QLoRA** (4-bit quantization + LoRA adapters). This version adds:
- ✅ Larger stratified dataset (300 records)
- ✅ Label masking (only train on assistant tokens, not prompt)
- ✅ Proper attention mask in inference
- ✅ Automated evaluation with per-tier breakdown & confusion matrix
- ✅ Hyperparameter sweep across 3 LoRA configs

---
Colab Link: https://colab.research.google.com/drive/1Cn0FzXXS-qM9Xf_gw_b2y5vDdXogNKDD#scrollTo=a27bdcbe

---
## Part 1: Setup

### 1.1 — Install Dependencies

Run the cell below to install all required packages. No changes needed here.

In [1]:
!pip install  -q --upgrade  bitsandbytes trl peft accelerate transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.8 MB/s eta 0:00:00


### 1.2 — Configuration

The training hyperparameters are set for you below. Read through them — you will be asked about some of these values in the reflection questions.

In [ ]:
import json
import re
import random
from enum import Enum
from typing import List
from pydantic import BaseModel, Field, field_validator
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
from google.colab import userdata

MODEL_NAME       = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH   = 1024


LORA_RANK        = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05
BATCH_SIZE       = 4
GRAD_ACCUM       = 2
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 5
SAVE_DIR         = "./transaction_risk_lora"
RANDOM_SEED      = 42

DATASET_SAMPLE_N = 300

SWEEP_CONFIGS = [
    {
        "name"        : "baseline",
        "lora_rank"   : 16,
        "lora_alpha"  : 32,
        "learning_rate": 2e-4,
        "num_epochs"  : 3,
        "description" : "Original config — low rank, 3 epochs",
    },
    {
        "name"        : "high_rank",
        "lora_rank"   : 32,
        "lora_alpha"  : 64,
        "learning_rate": 2e-4,
        "num_epochs"  : 5,
        "description" : "Higher rank + more epochs — more adapter capacity",
    },
    {
        "name"        : "lower_lr",
        "lora_rank"   : 32,
        "lora_alpha"  : 64,
        "learning_rate": 5e-5,
        "num_epochs"  : 5,
        "description" : "Same rank but 4× lower learning rate — less risk of overfitting",
    },
]

random.seed(RANDOM_SEED)
print("✅ Config ready")
print(f"   Dataset: {DATASET_SAMPLE_N} records | LoRA rank: {LORA_RANK} | α: {LORA_ALPHA} | epochs: {NUM_EPOCHS}")
print(f"   Sweep grid: {len(SWEEP_CONFIGS)} configs")

✅ Config ready
   Dataset: 300 records | LoRA rank: 32 | α: 64 | epochs: 5
   Sweep grid: 3 configs


### 1.3 — Define Pydantic Output Schema

In [ ]:
class RiskLevel(str, Enum):
    LOW    = "low_risk"
    MEDIUM = "medium_risk"
    HIGH   = "high_risk"
    CRITICAL = "critical_risk"


class RecommendedAction(str, Enum):
    ALLOW  = "allow"
    OTP    = "ask_otp"
    OTP_FLAGGED = "ask_otp_and_flag_for_review"
    BLOCK  = "block"
    REVIEW = "flag_for_review"


class TransactionRiskScore(BaseModel):
    fraud_probability_value: float = Field(
        ge=0.0, le=1.0,
        description="Probability of fraud, between 0.0 and 1.0"
    )

    fraud_probability_analysis: RiskLevel = Field(
        description="Risk tier: low_risk / medium_risk / high_risk / critical_risk"
    )
    action: RecommendedAction = Field(
        description="Recommended action to take on this transaction"
    )
    reasoning: str = Field(
        min_length=20,
        description="Plain-language explanation of why this score was assigned"
    )

    risk_factors: List[str] = Field(
        default_factory=list,
        description="Bullet-point list of key risk signals identified"
    )

    @field_validator("fraud_probability_value")
    @classmethod
    def round_probability(cls, v: float) -> float:
        return round(v, 4)

    def summary(self) -> str:
        return (
            f"[{self.fraud_probability_analysis.value.upper()}] "
            f"p={self.fraud_probability_value} → {self.action.value}"
        )


class TransactionInput(BaseModel):
    """Validates raw transaction features before they are sent to the model."""
    current_transaction_amount: float
    current_transaction_time: str
    current_transaction_time_gap_from_previous_transaction_time_in_minutes: int
    account_age_in_days: int
    average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month: float
    usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes: int
    usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month: str


test_obj = TransactionRiskScore(
    fraud_probability_value    = 0.72,
    fraud_probability_analysis = "high_risk",
    action                     = "ask_otp_and_flag_for_review",
    reasoning                  = "Amount is 10x the average and transaction occurred at 2am.",
    risk_factors               = ["Unusual hour", "Amount spike"]
)
print("✅ Pydantic schema works:", test_obj.summary())

✅ Pydantic schema works: [HIGH_RISK] p=0.72 → ask_otp_and_flag_for_review


### 1.4 — Login to HuggingFace

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("✅ HuggingFace login successful")

✅ HuggingFace login successful


---
## Part 2: Data

### 2.1 — Load the Dataset



In [5]:
raw_dataset = load_dataset(
    "peacemakerzvidzanayi/aml_fraud_detection_and_reasoning_50k_syntetic",
    split="train"
)
df = raw_dataset.to_pandas()

print(f"Dataset loaded: {len(raw_dataset):,} records")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nRisk tier distribution:")
print(df["fraud_probability_analysis"].value_counts().to_string())
print("\nSample record:")
print(json.dumps(df.iloc[0].to_dict(), indent=2, default=str))

nbs_aml_fraud_detection_50k_dataset_NOW_(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset loaded: 50,000 records
Columns: ['current_transaction_amount', 'current_transaction_time', 'current_transaction_time_gap_from_previous_transaction_time_in_minutes', 'account_age_in_days', 'average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month', 'usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes', 'usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month', 'fraud_probability_value', 'fraud_probability_analysis', 'action', 'reasoning']

Risk tier distribution:
fraud_probability_analysis
medium_risk      25884
low_risk         17697
high_risk         6246
critical_risk      173

Sample record:
{
  "current_transaction_amount": 350,
  "current_transaction_time": "2026-03-10 19:00:38",
  "current_transaction_time_gap_from_previous_transaction_time_in_minutes": 214,
  "account_age_in_days": 187,
  "average_transaction_amount_in_the_analysis_in_th

### 2.2 — Stratified Sampling

We take **300 records** from the full 50k dataset using **stratified sampling** to ensure the model
sees a balanced mix of risk tiers during training. We oversample  since it represents
only ~0.3% of the real dataset (173 out of 50k records) — without oversampling, the model never
learns to identify it.


In [ ]:
from datasets import Dataset

# ── Stratified strata ─────────────────────────────────────────────
STRATA = {
    "low_risk"    : 75,
    "medium_risk" : 105,
    "high_risk"   : 75,
    "critical_risk": 45,   
}

sampled_frames = []
for tier, n in STRATA.items():
    tier_df  = df[df["fraud_probability_analysis"] == tier]
    n_actual = min(n, len(tier_df))
    sampled_frames.append(tier_df.sample(n=n_actual, random_state=RANDOM_SEED))
    print(f"  Sampled {n_actual} × '{tier}'")

sampled_df = (
    pd.concat(sampled_frames)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
total = len(sampled_df)
print(f"\nFinal sample: {total} records")
print(f"Distribution:\n{sampled_df['fraud_probability_analysis'].value_counts().to_string()}")

  Sampled 75 × 'low_risk'
  Sampled 105 × 'medium_risk'
  Sampled 75 × 'high_risk'
  Sampled 45 × 'critical_risk'

Final sample: 300 records
Distribution:
fraud_probability_analysis
medium_risk      105
high_risk         75
low_risk          75
critical_risk     45


### 2.3 — Format Data as Chat Instructions

We convert each record into a three-turn conversation: a **system** prompt describing the task, a **user** message with the transaction features, and an **assistant** message containing the expected JSON output.


In [7]:
SYSTEM_PROMPT = """You are an expert AML (Anti-Money Laundering) transaction risk analyst.
Given transaction features, you output a structured JSON risk assessment.
Your JSON must match this exact schema:
{
  "fraud_probability_value": <float 0.0-1.0>,
  "fraud_probability_analysis": <"low_risk" | "medium_risk" | "high_risk" | "critical_risk">,
  "action": <"allow" | "ask_otp" | "ask_otp_and_flag_for_review" | "block" | "flag_for_review">,
  "reasoning": <string explaining the score>,
  "risk_factors": [<list of key risk signals>]
}
Output ONLY the JSON object. No preamble, no markdown, no explanation outside the JSON."""


def record_to_user_prompt(row: dict) -> str:
    """Formats the transaction fields into a structured user message."""
    return f"""Analyse this transaction and return a risk assessment JSON:

Transaction Amount   : {row['current_transaction_amount']}
Transaction Time     : {row['current_transaction_time']}
Time Gap (mins)      : {row['current_transaction_time_gap_from_previous_transaction_time_in_minutes']}
Account Age (days)   : {row['account_age_in_days']}
Avg Amount (3m)      : {row['average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month']:.2f}
Usual Smallest Gap   : {row['usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes']} mins
Usual Time Window    : {row['usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month']}"""


def record_to_assistant_response(row: dict) -> str:
    """Builds the expected assistant JSON output from the ground-truth labels."""
    reasoning    = str(row.get("reasoning", ""))
    raw_factors  = [f.strip() for f in re.split(r'\s+and\s+', reasoning) if len(f.strip()) > 10]
    risk_factors = raw_factors[:4] if raw_factors else ["See reasoning for details"]
    output = {
        "fraud_probability_value"   : round(float(row["fraud_probability_value"]), 4),
        "fraud_probability_analysis": row["fraud_probability_analysis"],
        "action"                    : row["action"],
        "reasoning"                 : reasoning,
        "risk_factors"              : risk_factors,
    }
    return json.dumps(output, indent=2)


def format_as_chat(row: dict) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": record_to_user_prompt(row)},
            {"role": "assistant", "content": record_to_assistant_response(row)},
        ]
    }


formatted   = [format_as_chat(r) for r in sampled_df.to_dict(orient="records")]
hf_dataset  = Dataset.from_list(formatted)
split       = hf_dataset.train_test_split(test_size=0.15, seed=RANDOM_SEED)  # ~45 eval records
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples | Eval: {len(eval_dataset)} examples")
print("\nFirst training example — system turn:")
print(train_dataset[0]["messages"][0]["content"])
print("\nFirst training example — user turn:")
print(train_dataset[0]["messages"][1]["content"])

Train: 255 examples | Eval: 45 examples

First training example — system turn:
You are an expert AML (Anti-Money Laundering) transaction risk analyst.
Given transaction features, you output a structured JSON risk assessment.
Your JSON must match this exact schema:
{
  "fraud_probability_value": <float 0.0-1.0>,
  "fraud_probability_analysis": <"low_risk" | "medium_risk" | "high_risk" | "critical_risk">,
  "action": <"allow" | "ask_otp" | "ask_otp_and_flag_for_review" | "block" | "flag_for_review">,
  "reasoning": <string explaining the score>,
  "risk_factors": [<list of key risk signals>]
}
Output ONLY the JSON object. No preamble, no markdown, no explanation outside the JSON.

First training example — user turn:
Analyse this transaction and return a risk assessment JSON:

Transaction Amount   : 788
Transaction Time     : 2026-03-10 03:32:59
Time Gap (mins)      : 27
Account Age (days)   : 886
Avg Amount (3m)      : 221.70
Usual Smallest Gap   : 28 mins
Usual Time Window    : 06:50:23

---
## Part 3: Model Setup

### 3.1 — Load the Base Model in 4-bit

We use `BitsAndBytesConfig` to load the model in 4-bit quantization to reduces VRAM from ~8 GB to ~2 GB, making training possible on a free Colab T4.


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token            = hf_token,
    padding_side     = "right",
    model_max_length = MAX_SEQ_LENGTH,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = hf_token,
    torch_dtype         = torch.float16,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    used  = torch.cuda.memory_allocated(0) / 1e9
    print(f"✅ Model loaded | VRAM: {used:.2f} GB used / {total:.1f} GB total")
else:
    print("⚠️  No GPU — switch to T4 runtime before continuing")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded | VRAM: 1.03 GB used / 15.6 GB total


### 3.2 — Attach LoRA Adapters
Attach LORA adapters using `peft` librarry

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = LORA_RANK,
    lora_alpha   = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias         = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable (LoRA)    : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Frozen (base model) : {total-trainable:,}  ({100*(1-trainable/total):.2f}%)")

Total parameters    : 771,819,520
Trainable (LoRA)    : 22,544,384  (2.92%)
Frozen (base model) : 749,275,136  (97.08%)


---
## Part 4: Training

### 4.1 — Tokenize the Dataset

Convert conversation into token IDs using the model's tokenizer.


In [ ]:
# ── Label Masking ──────────


def find_assistant_start(input_ids: list, tokenizer) -> int:
    """Return the index of the first token belonging to the assistant response.

    Llama-3 chat template ends the assistant header with the token sequence
    for '<|start_header_id|>assistant<|end_header_id|>\n\n'.
    We search for the last occurrence of that pattern.
    """
    # Encode the assistant header marker
    header = tokenizer.encode(
        "<|start_header_id|>assistant<|end_header_id|>\n\n",
        add_special_tokens=False
    )
    n = len(header)
    # Scan from the right to find the last occurrence
    for i in range(len(input_ids) - n, -1, -1):
        if input_ids[i:i+n] == header:
            return i + n  
    return 0   


def tokenize_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    input_ids = tokenized["input_ids"]
    labels    = input_ids.copy()

    # Mask prompt tokens — set to -100 so CrossEntropyLoss ignores them
    assistant_start = find_assistant_start(input_ids, tokenizer)
    for i in range(assistant_start):
        labels[i] = -100

    tokenized["labels"] = labels
    return tokenized


train_tokenized = train_dataset.map(
    tokenize_example,
    remove_columns = train_dataset.column_names,
    desc           = "Tokenizing train",
)
eval_tokenized = eval_dataset.map(
    tokenize_example,
    remove_columns = eval_dataset.column_names,
    desc           = "Tokenizing eval",
)

# Sanity-check: count how many tokens are masked vs trained
sample_labels = train_tokenized[0]["labels"]
masked  = sum(1 for l in sample_labels if l == -100)
trained = sum(1 for l in sample_labels if l != -100)
print(f"✅ Tokenized | Train: {len(train_tokenized)} | Eval: {len(eval_tokenized)}")
print(f"   Sample token breakdown — masked (prompt): {masked} | trained (response): {trained}")

Tokenizing train:   0%|          | 0/255 [00:00<?, ? examples/s]

Tokenizing eval:   0%|          | 0/45 [00:00<?, ? examples/s]

✅ Tokenized | Train: 255 | Eval: 45
   Sample token breakdown — masked (prompt): 302 | trained (response): 281


### 4.2 — Configure and Run Training

We use `TrainingArguments` from `transformers` and `SFTTrainer` from `trl`.


In [ ]:
import os
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
from google.colab import drive

# ── Mount Google Drive for checkpoint persistence ─────────────────────────
# This ensures they survive disconnections.

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/transaction_risk_lora_r32"
os.makedirs(SAVE_DIR, exist_ok=True)


training_args = TrainingArguments(
    output_dir                  = SAVE_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = NUM_EPOCHS,
    warmup_steps                = 5,
    learning_rate               = LEARNING_RATE,
    fp16                        = False,
    bf16                        = True,
    logging_steps               = 5,
    eval_strategy               = "steps",   
    eval_steps                  = 50,
    save_strategy               = "steps",   
    save_steps                  = 50,
    save_total_limit            = 2,         
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    optim                       = "paged_adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = RANDOM_SEED,
    report_to                   = "none",
    remove_unused_columns       = False,
)


collator = DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of = 8,
    return_tensors     = "pt",
    padding            = True,
)

trainer = SFTTrainer(
    model         = model,
    data_collator = collator,
    train_dataset = train_tokenized,
    eval_dataset  = eval_tokenized,
    args          = training_args,
)


# If training crashed previously, this picks up from the last saved step.
# If no checkpoint exists, it starts from scratch automatically.
last_checkpoint = None
if os.path.isdir(SAVE_DIR):
    checkpoints = [
        os.path.join(SAVE_DIR, d) for d in os.listdir(SAVE_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"▶️  Resuming from: {last_checkpoint}")
    else:
        print("▶️  No checkpoint found — starting from scratch")

# ── Train ─────────────────────────────────────────────────────────────────
print("Starting training...")
trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"\n✅ Done | Steps: {trainer_stats.global_step} | Loss: {trainer_stats.training_loss:.4f} | Time: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Truncating train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

▶️  No checkpoint found — starting from scratch
Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.046495,0.046812
100,0.041098,0.042806
150,0.036468,0.042564


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


✅ Done | Steps: 160 | Loss: 0.0884 | Time: 2990s


### 4.3 — Save the Model

Run the cell below to save your LoRA adapter weights to disk.

In [13]:
import os
from huggingface_hub import HfApi

# ── Save locally first ────────────────────────────────────────────────────
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_DIR, f))
    for f in os.listdir(SAVE_DIR)
    if os.path.isfile(os.path.join(SAVE_DIR, f))
) / 1e6

print(f"✅ Saved locally to '{SAVE_DIR}' ({adapter_size:.1f} MB)")
print(f"   Files: {[f for f in os.listdir(SAVE_DIR) if not f.startswith('checkpoint')]}")

HF_USERNAME = "johnmboga"
HF_REPO_NAME = f"{HF_USERNAME}/transaction-risk-lora-r32"

model.push_to_hub(
    HF_REPO_NAME,
    token      = hf_token,
    private    = False,
    commit_message = "Add LoRA adapter fine-tuned on AML fraud detection dataset",
)
tokenizer.push_to_hub(
    HF_REPO_NAME,
    token      = hf_token,
    private    = True,
)

print(f"\n✅ Pushed to HuggingFace: https://huggingface.co/{HF_REPO_NAME}")
print(f"   Adapter only — base model ({MODEL_NAME}) is referenced, not duplicated")

✅ Saved locally to '/content/drive/MyDrive/transaction_risk_lora_r32' (107.4 MB)
   Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         | 1.21MB / 90.2MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp_5f7ge31/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            


✅ Pushed to HuggingFace: https://huggingface.co/johnmboga/transaction-risk-lora-r32
   Adapter only — base model (meta-llama/Llama-3.2-1B-Instruct) is referenced, not duplicated


---
## Part 5: Inference & Evaluation

### 5.1 — Reload and Run Inference




In [ ]:
from peft import PeftModel
from pydantic import ValidationError

HF_USERNAME  = "johnmboga"
HF_REPO_NAME = f"{HF_USERNAME}/transaction-risk-lora-r32"


base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", token=hf_token, torch_dtype=torch.bfloat16,
)
ft_tokenizer = AutoTokenizer.from_pretrained(HF_REPO_NAME, token=hf_token)
ft_model     = PeftModel.from_pretrained(base, HF_REPO_NAME, token=hf_token)
ft_model.eval()
print("✅ Fine-tuned model loaded from HuggingFace")


def run_inference(transaction: dict, mdl=None, tok=None, max_new_tokens: int = 400) -> str:
    mdl = mdl or ft_model
    tok = tok or ft_tokenizer
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": record_to_user_prompt(transaction)},
    ]

    prompt = tok.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize              = False,
    )


    inputs       = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    input_ids    = inputs.input_ids.to(mdl.device)
    attn_mask    = inputs.attention_mask.to(mdl.device)

    with torch.no_grad():
        outputs = mdl.generate(
            input_ids,
            attention_mask = attn_mask,
            max_new_tokens = max_new_tokens,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tok.eos_token_id,
        )
    new_tokens = outputs[0][input_ids.shape[-1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


def parse_risk_response(raw_output: str) -> TransactionRiskScore:
    clean      = re.sub(r'```(?:json)?', '', raw_output).strip()
    json_match = re.search(r'\{.*\}', clean, re.DOTALL)
    if not json_match:
        raise ValueError(f"No JSON object found in model output:\n{raw_output[:300]}")
    data = json.loads(json_match.group())
    return TransactionRiskScore(**data)


def score_transaction(transaction: dict, mdl=None, tok=None):
    raw = run_inference(transaction, mdl=mdl, tok=tok)
    try:
        return parse_risk_response(raw), raw
    except (json.JSONDecodeError, ValidationError, ValueError) as e:
        print(f"⚠️  Parse error: {e}")
        return None, raw


test_record = sampled_df.iloc[5].to_dict()
score, raw  = score_transaction(test_record)
if score:
    print(f"\nTest passed ✅ → {score.summary()}")
else:
    print(f"\nParse failed ❌ — check parse_risk_response")
    print(f"Raw output: {raw[:300]}")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/90.2M [00:00<?, ?B/s]

✅ Fine-tuned model loaded from HuggingFace

Test passed ✅ → [CRITICAL_RISK] p=0.811 → block


### 5.2 — Before vs After Comparison

Run the same transaction through the **base model** and the **fine-tuned model** and compare their outputs side-by-side.

In [16]:
print("Loading base model for comparison...")
base_model_raw = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", token=hf_token, torch_dtype=torch.float16,
)
base_tok_raw = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, padding_side="right")
if base_tok_raw.pad_token is None:
    base_tok_raw.pad_token = base_tok_raw.eos_token
base_model_raw.eval()

test_tx = sampled_df.iloc[0].to_dict()
print(f"\nTransaction: amount={test_tx['current_transaction_amount']} | expected={test_tx['fraud_probability_analysis']} → {test_tx['action']}")

print("\n" + "─" * 30 + " BASE MODEL " + "─" * 30)
base_raw = run_inference(test_tx, mdl=base_model_raw, tok=base_tok_raw)
print(base_raw[:600])
try:
    base_score = parse_risk_response(base_raw)
    print(f"\nParsed ✅ → {base_score.summary()}")
except Exception as e:
    print(f"\nPydantic validation failed ❌: {e}")

print("\n" + "─" * 28 + " FINE-TUNED MODEL " + "─" * 28)
ft_score, ft_raw = score_transaction(test_tx)
print(ft_raw[:600])
if ft_score:
    print(f"\nParsed ✅ → {ft_score.summary()}")
    print(f"Correct tier  : {'✅' if ft_score.fraud_probability_analysis.value == test_tx['fraud_probability_analysis'] else '❌'}")
    print(f"Correct action: {'✅' if ft_score.action.value == test_tx['action'] else '❌'}")

Loading base model for comparison...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]


Transaction: amount=730 | expected=high_risk → ask_otp_and_flag_for_review

────────────────────────────── BASE MODEL ──────────────────────────────
{
  "fraud_probability_value": 0.0,
  "fraud_probability_analysis": "low_risk",
  "action": "block",
  "reasoning": "Transaction amount is less than $100,000 and time gap is less than 30 minutes, account age is less than 30 days, avg amount is less than $50,000, and usual smallest gap is less than 7 minutes.",
  "risk_factors": ["low_risk"]
}

Parsed ✅ → [LOW_RISK] p=0.0 → block

──────────────────────────── FINE-TUNED MODEL ────────────────────────────
{
  "fraud_probability_value": 0.811,
  "fraud_probability_analysis": "critical_risk",
  "action": "block",
  "reasoning": "current_transaction_amount is significantly higher than average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month which strongly indicates potential fraudulent activity and current_transaction_time falls outside usual_time_range_of_accoun

### 5.3 — Automated Evaluation: Per-Tier Accuracy & Confusion Matrix

In [ ]:
from collections import defaultdict

TIERS = ["low_risk", "medium_risk", "high_risk", "critical_risk"]


def evaluate_model(model, tokenizer, eval_records: list, label: str = "Model") -> dict:
    """
    Run eval_records through the model and return a metrics dict with:
      - tier_accuracy, action_accuracy, parse_rate
      - per_tier_acc  : {tier: acc}
      - confusion_matrix : {actual_tier: {pred_tier: count}}
    Also prints a formatted report.
    """
    results        = []
    parse_failures = 0
    conf_matrix    = {t: defaultdict(int) for t in TIERS}
    tier_totals    = defaultdict(int)
    tier_correct   = defaultdict(int)

    for i, record in enumerate(eval_records):
        actual_tier   = record["fraud_probability_analysis"]
        actual_action = record["action"]
        tier_totals[actual_tier] += 1

        score, _ = score_transaction(record, mdl=model, tok=tokenizer)

        if score is None:
            parse_failures += 1
            conf_matrix[actual_tier]["parse_fail"] += 1
            results.append({"tier_match": False, "action_match": False})
            pred_tier = "parse_fail"
        else:
            pred_tier    = score.fraud_probability_analysis.value
            tier_match   = pred_tier == actual_tier
            action_match = score.action.value == actual_action
            if tier_match:
                tier_correct[actual_tier] += 1
            conf_matrix[actual_tier][pred_tier] += 1
            results.append({"tier_match": tier_match, "action_match": action_match})

        # Live progress
        mark = "✅" if results[-1]["tier_match"] else "❌"
        print(f"  [{i+1:03d}] actual={actual_tier:<13} predicted={pred_tier:<13} {mark}")

    n = len(results)
    tier_acc   = sum(r["tier_match"]   for r in results) / n
    action_acc = sum(r["action_match"] for r in results) / n
    parse_rate = (n - parse_failures) / n

    # ── Per-tier accuracy ────────────────────────────────────────
    per_tier_acc = {
        t: (tier_correct[t] / tier_totals[t]) if tier_totals[t] > 0 else None
        for t in TIERS
    }

    # ── Print report ─────────────────────────────────────────────
    w = 44
    print(f"\n{'─'*w}")
    print(f"  {label}")
    print(f"{'─'*w}")
    print(f"  Total records      : {n}")
    print(f"  Risk Tier Accuracy : {tier_acc*100:.1f}%")
    print(f"  Action Accuracy    : {action_acc*100:.1f}%")
    print(f"  Pydantic Pass Rate : {parse_rate*100:.1f}%")
    print(f"\n  Per-tier accuracy:")
    for t in TIERS:
        acc = per_tier_acc[t]
        if acc is None:
            print(f"    {t:<15}: n/a (no examples)")
        else:
            bar = '█' * int(acc * 20)
            print(f"    {t:<15}: {acc*100:5.1f}%  {bar}")

    # ── Confusion matrix ─────────────────────────────────────────
    col_w = 13
    header = f"  {'Actual \\ Pred':<15}" + "".join(f"{t[:12]:<{col_w}}" for t in TIERS)
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    print(f"  {'─'*len(header)}")
    print(header)
    print(f"  {'─'*len(header)}")
    for actual in TIERS:
        row = f"  {actual:<15}"
        for pred in TIERS:
            count = conf_matrix[actual][pred]
            row  += f"{count:<{col_w}}"
        print(row)
    print(f"  {'─'*w}")

    return {
        "tier_accuracy"  : tier_acc,
        "action_accuracy": action_acc,
        "parse_rate"     : parse_rate,
        "per_tier_acc"   : per_tier_acc,
        "confusion_matrix": dict(conf_matrix),
    }



eval_records = eval_dataset.to_list()  

eval_raw_records = []


eval_n = max(1, int(len(sampled_df) * 0.15))
eval_raw = sampled_df.tail(eval_n).to_dict(orient="records")

print(f"Running evaluation on {len(eval_raw)} held-out records...\n")
ft_metrics = evaluate_model(ft_model, ft_tokenizer, eval_raw, label="Fine-Tuned Model")

Running evaluation on 45 held-out records...

  [001] actual=medium_risk   predicted=critical_risk ❌
  [002] actual=low_risk      predicted=critical_risk ❌
  [003] actual=medium_risk   predicted=critical_risk ❌
  [004] actual=high_risk     predicted=critical_risk ❌
  [005] actual=low_risk      predicted=critical_risk ❌
  [006] actual=medium_risk   predicted=low_risk      ❌
  [007] actual=low_risk      predicted=critical_risk ❌
  [008] actual=high_risk     predicted=critical_risk ❌
  [009] actual=low_risk      predicted=critical_risk ❌
  [010] actual=medium_risk   predicted=low_risk      ❌
  [011] actual=medium_risk   predicted=critical_risk ❌
  [012] actual=critical_risk predicted=critical_risk ✅
  [013] actual=medium_risk   predicted=critical_risk ❌
  [014] actual=critical_risk predicted=critical_risk ✅
  [015] actual=high_risk     predicted=low_risk      ❌
  [016] actual=low_risk      predicted=low_risk      ✅
  [017] actual=low_risk      predicted=critical_risk ❌
  [018] actual=medi

---
## Part 6: Hyperparameter Sweep

We compare three configs:
- `baseline` — rank=16, α=32, lr=2e-4, 3 epochs (original)
- `high_rank` — rank=32, α=64, lr=2e-4, 5 epochs (more capacity)
- `lower_lr`  — rank=32, α=64, lr=5e-5, 5 epochs (slower, less overfitting)

In [ ]:
import gc
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer

sweep_results = []


def run_sweep_config(cfg: dict, train_tok, eval_tok, base_model_name: str, bnb_cfg, tok, hf_tok):
    """Load a fresh model, attach LoRA with cfg, train, evaluate, return metrics."""
    print(f"\n{'='*55}")
    print(f"  Config: {cfg['name']} — {cfg['description']}")
    print(f"  rank={cfg['lora_rank']}, α={cfg['lora_alpha']}, lr={cfg['learning_rate']}, epochs={cfg['num_epochs']}")
    print(f"{'='*55}")

    # ── Load fresh base model ────────────────────────────────────
    sweep_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, quantization_config=bnb_cfg,
        device_map="auto", token=hf_tok, torch_dtype=torch.bfloat16,
    )
    sweep_model.config.use_cache      = False
    sweep_model.config.pretraining_tp = 1
    sweep_model = prepare_model_for_kbit_training(sweep_model, use_gradient_checkpointing=True)

    # ── Attach LoRA ───────────────────────────────────────────────
    lora_cfg = LoraConfig(
        task_type    = TaskType.CAUSAL_LM,
        r            = cfg["lora_rank"],
        lora_alpha   = cfg["lora_alpha"],
        lora_dropout = LORA_DROPOUT,
        bias         = "none",
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    )
    sweep_model = get_peft_model(sweep_model, lora_cfg)

    # ── Training args ─────────────────────────────────────────────
    sweep_dir = f"/content/drive/MyDrive/sweep_{cfg['name']}"
    import os; os.makedirs(sweep_dir, exist_ok=True)

    t_args = TrainingArguments(
        output_dir                  = sweep_dir,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = cfg["num_epochs"],
        warmup_steps                = 5,
        learning_rate               = cfg["learning_rate"],
        fp16=False, bf16=True,
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 1,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        optim                       = "paged_adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = RANDOM_SEED,
        report_to                   = "none",
        remove_unused_columns       = False,
    )

    collator = DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors="pt", padding=True)
    trainer  = SFTTrainer(
        model=sweep_model, data_collator=collator,
        train_dataset=train_tok, eval_dataset=eval_tok, args=t_args,
    )

    stats = trainer.train()
    eval_metrics = trainer.evaluate()
    eval_loss    = eval_metrics.get("eval_loss", float("nan"))
    print(f"  ✅ Training done | train_loss={stats.training_loss:.4f} | eval_loss={eval_loss:.4f}")

    # ── Quick accuracy on 20 eval records ────────────────────────
    sweep_model.eval()
    quick_eval  = sampled_df.tail(20).to_dict(orient="records")
    correct = 0
    for rec in quick_eval:
        s, _ = score_transaction(rec, mdl=sweep_model, tok=tok)
        if s and s.fraud_probability_analysis.value == rec["fraud_probability_analysis"]:
            correct += 1
    quick_acc = correct / len(quick_eval)
    print(f"  Quick accuracy (20 records): {quick_acc*100:.1f}%")

    result = {
        "name"      : cfg["name"],
        "config"    : cfg,
        "train_loss": stats.training_loss,
        "eval_loss" : eval_loss,
        "quick_acc" : quick_acc,
    }

    # ── Free GPU memory before next run ──────────────────────────
    del sweep_model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return result


# ── Run the sweep ─────────────────────────────────────────────────
CONFIGS_TO_RUN = SWEEP_CONFIGS

for cfg in CONFIGS_TO_RUN:
    res = run_sweep_config(
        cfg, train_tokenized, eval_tokenized,
        MODEL_NAME, bnb_config, tokenizer, hf_token
    )
    sweep_results.append(res)

# ── Comparison table ─────────────────────────────────────────────
print("\n" + "="*65)
print("  SWEEP RESULTS SUMMARY")
print("="*65)
print(f"  {'Config':<12} {'rank':>5} {'α':>5} {'lr':>8} {'epochs':>7} {'train_loss':>11} {'eval_loss':>10} {'acc@20':>7}")
print("-"*65)
for r in sweep_results:
    c = r["config"]
    print(f"  {r['name']:<12} {c['lora_rank']:>5} {c['lora_alpha']:>5} {c['learning_rate']:>8.0e} "
          f"{c['num_epochs']:>7} {r['train_loss']:>11.4f} {r['eval_loss']:>10.4f} {r['quick_acc']*100:>6.1f}%")

if sweep_results:
    best = min(sweep_results, key=lambda x: x["eval_loss"])
    print(f"\n  🏆 Best config by eval_loss: '{best['name']}' ({best['eval_loss']:.4f})")


  Config: baseline — Original config — low rank, 3 epochs
  rank=16, α=32, lr=0.0002, epochs=3


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [19]:
# Now let's add two more test transactions
high_risk_transaction = {
    "current_transaction_amount": 8500,
    "current_transaction_time": "02:14:00",
    "current_transaction_time_gap_from_previous_transaction_time_in_minutes": 3,
    "account_age_in_days": 12,
    "average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month": 120.0,
    "usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes": 45,
    "usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month": "09:00:00 to 18:30:00"
}

low_risk_transaction = {
    "current_transaction_amount": 150,
    "current_transaction_time": "14:30:00",
    "current_transaction_time_gap_from_previous_transaction_time_in_minutes": 120,
    "account_age_in_days": 400,
    "average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month": 180.0,
    "usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes": 60,
    "usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month": "09:00:00 to 17:00:00"
}

for label, tx in [("HIGH RISK", high_risk_transaction), ("LOW RISK", low_risk_transaction)]:
    print(f"\n{'='*55}")
    print(f"  My {label} transaction")
    print(f"{'='*55}")

    try:
        TransactionInput(**tx)
        print("Input validation: ✅")
    except Exception as e:
        print(f"Input validation: ❌ {e}")
        continue

    score, raw = score_transaction(tx)
    if score:
        print(f"Risk tier  : {score.fraud_probability_analysis.value}")
        print(f"Action     : {score.action.value}")
        print(f"Score      : {score.fraud_probability_value}")
        print(f"Reasoning  : {score.reasoning}")
        print(f"Risk factors:")
        for rf in score.risk_factors:
            print(f"  • {rf}")
    else:
        print(f"❌ Parse failed. Raw: {raw[:200]}")


  My HIGH RISK transaction
Input validation: ✅
Risk tier  : critical_risk
Action     : block
Score      : 0.811
Reasoning  : current_transaction_amount is significantly higher than average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month which strongly indicates potential fraudulent activity and current_transaction_time falls outside usual_time_range_of_account_transactions_in_the_analysis_in_the_past_3_months_including_current_month which is a significant anomaly and current_transaction_time_gap_from_previous_transaction_time_in_minutes is much smaller than usual_smallest_transaction_time_gap_when_busy_analysis_in_the_past_3_months_including_current_month_in_minutes suggesting rapid suspicious activity and account_age_in_days reflects an established account with reasonable history.
Risk factors:
  • current_transaction_amount is significantly higher than average_transaction_amount_in_the_analysis_in_the_past_3_months_including_current_month which strong